In [3]:
import ast
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Define path to the enriched dataset
csv_path = Path("../data/processed/adzuna_jobs_with_skills.csv")
# Load the dataset
df = pd.read_csv(csv_path)

#df.head(2)

df["created"] = pd.to_datetime(df["created"])
df["skills"] = df["skills"].apply(lambda x : ast.literal_eval(x) if pd.notnull(x) else [])
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   salary_max      258 non-null    float64            
 1   latitude        858 non-null    float64            
 2   redirect_url    1025 non-null   str                
 3   title           1025 non-null   str                
 4   created         1025 non-null   datetime64[us, UTC]
 5   salary_min      258 non-null    float64            
 6   id              1025 non-null   int64              
 7   description     1025 non-null   str                
 8   longitude       858 non-null    float64            
 9   contract_time   417 non-null    str                
 10  _search_query   1025 non-null   str                
 11  company         1024 non-null   str                
 12  category        1025 non-null   str                
 13  category_tag    1025 non-null   str         

In [5]:
type(df["skills"].iloc[0])

list

In [6]:
# Coherence check : how many rows have salary max but not salary min, and vice versa
mismatch = df[df["salary_min"].notna() != df["salary_max"].notna()]
print(f"Rows with mismatched salary info: {len(mismatch)}")
# Build boolean mask + create independent subset of jobs with salary info
mask = df["salary_min"].notna() & df["salary_max"].notna()
df_salary = df[mask].copy()
print(f"df_salary shape: {df_salary.shape}")

Rows with mismatched salary info: 0
df_salary shape: (258, 23)


In [7]:
df_salary[["salary_min", "salary_max"]].describe()

,salary_min,salary_max
count,258.000000,258.000000
mean,98566.337209,129813.496124
std,36706.588717,48318.396782
min,41600.000000,52000.000000
25%,72000.000000,95000.000000
50%,89050.000000,120000.000000
75%,120000.000000,150000.000000
max,228000.000000,313500.000000


In [8]:
df_salary["salary_avg"] = (df_salary["salary_min"] + df_salary["salary_max"]) / 2  


In [9]:
df_salary[["salary_min", "salary_max", "salary_avg"]].head()

,salary_min,salary_max,salary_avg
0,78777.0,98887.0,88832.0
2,110000.0,140000.0,125000.0
3,54059.0,83179.0,68619.0
6,83200.0,93600.0,88400.0
8,68000.0,78000.0,73000.0


In [10]:
df_salary[["salary_min", "salary_max", "salary_avg"]].describe()

,salary_min,salary_max,salary_avg
count,258.000000,258.000000,258.000000
mean,98566.337209,129813.496124,114189.916667
std,36706.588717,48318.396782,41128.029702
min,41600.000000,52000.000000,46800.000000
25%,72000.000000,95000.000000,85000.000000
50%,89050.000000,120000.000000,105000.000000
75%,120000.000000,150000.000000,134790.000000
max,228000.000000,313500.000000,270750.000000


In [11]:
salary_summary = df_salary.groupby("category_tag")["salary_avg"].agg([
    "count", "mean", "median", "min", "max"
]).round(0)
salary_summary


,count,mean,median,min,max
category_tag,,,,,
accounting-finance-jobs,24,101336.0,91746.0,52500.0,228800.0
admin-jobs,1,70000.0,70000.0,70000.0,70000.0
consultancy-jobs,3,70987.0,62730.0,62730.0,87500.0
creative-design-jobs,1,122500.0,122500.0,122500.0,122500.0
engineering-jobs,13,135055.0,118200.0,55120.0,270750.0
graduate-jobs,1,79750.0,79750.0,79750.0,79750.0
healthcare-nursing-jobs,1,87850.0,87850.0,87850.0,87850.0
hr-jobs,1,95981.0,95981.0,95981.0,95981.0
it-jobs,192,113384.0,104878.0,46800.0,232000.0


In [14]:
salary_summary = (
    df_salary
    .groupby("_search_query")["salary_avg"]
    .agg(["count", "median", "mean", "min", "max"])
    .round(0)
    .sort_values("count", ascending=False)
)

salary_summary

,count,median,mean,min,max
_search_query,,,,,
data analyst,68,89938.0,93452.0,46800.0,209500.0
business analyst,60,92446.0,103015.0,55000.0,228800.0
data engineer,47,120000.0,124577.0,55120.0,270750.0
data scientist,44,130000.0,129393.0,69582.0,215000.0
data consultant,35,137500.0,141785.0,52500.0,232000.0
BI analyst,4,107400.0,103620.0,74880.0,124800.0


In [16]:
print(f"Total: {salary_summary['count'].sum()}")  # should be 258

Total: 258


In [17]:
# Inspect the highest-paid "data analyst" postings
df_salary[df_salary["_search_query"] == "data analyst"].nlargest(5, "salary_avg")[
    ["title", "company", "city", "salary_avg"]
]

,title,company,city,salary_avg
203,Evaluation Lead,Waabi,Toronto,209500.0
174,25-199 - Data Engineer,Morson,Durham region,199680.0
175,25-199 - Data Engineer - Azure Databricks - 11...,CorGTA,Durham region,135200.0
145,"Senior Technical Analyst, Data Engineering and...",Fidelity International,Toronto,135000.0
73,"Senior Data Analyst, Business Intelligence",Turo,Toronto,128500.0


In [18]:
# How many df_salary rows have a city?
print(f"df_salary with city: {df_salary['city'].notna().sum()} / {len(df_salary)}")
print(f"\nUnique cities: {df_salary['city'].nunique()}")
print(f"\nTop 10 cities by frequency:")
print(df_salary["city"].value_counts().head(10))

df_salary with city: 250 / 258

Unique cities: 17

Top 10 cities by frequency:
city
Toronto            151
Peel region         27
Ottawa region       17
York region         13
Halton               9
Waterloo region      9
Middlesex            6
Durham region        5
Hamilton region      3
Elgin region         2
Name: count, dtype: int64


In [19]:
# Define the groups we want to keep as named categories

TOP_CITIES = ["Toronto", "Peel region","Ottawa region", "York region"]
def group_cities(city):
    if pd.isna(city):
        return "Unknown"
    elif city in TOP_CITIES:
        return city
    else:
        return "Other"
df_salary["city_grouped"] = df_salary["city"].apply(group_cities)

#Sanity check
print(df_salary["city_grouped"].value_counts())

city_grouped
Toronto          151
Other             42
Peel region       27
Ottawa region     17
York region       13
Unknown            8
Name: count, dtype: int64


In [20]:
salary_by_city = (
    df_salary[df_salary["city_grouped"] != "Unknown"]  
    .groupby("city_grouped")["salary_avg"]
    .agg(["count", "median", "mean", "min", "max"])
    .round(0)
    .sort_values("count", ascending=False)
)

salary_by_city

,count,median,mean,min,max
city_grouped,,,,,
Toronto,151,108000.0,117115.0,52500.0,270750.0
Other,42,98000.0,112089.0,48000.0,225000.0
Peel region,27,89702.0,101546.0,65000.0,225000.0
Ottawa region,17,112650.0,121062.0,46800.0,225000.0
York region,13,89877.0,89513.0,55000.0,120000.0


In [26]:
# Keep only postings where at least one skill was extracted
df_salary_skills = df_salary[df_salary["skills"].apply(len) > 0].copy()

print(f"df_salary_skills shape: {df_salary_skills.shape}")
print(f"Average skills per posting: {df_salary_skills['skills'].apply(len).mean():.1f}")

df_salary_skills shape: (61, 25)
Average skills per posting: 1.6


In [27]:
# Explode the skills list into one row per (posting, skill) pair
df_exploded_salary = df_salary_skills.explode("skills").reset_index(drop=True)

print(f"After explode: {len(df_exploded_salary)} rows")

After explode: 99 rows


In [28]:
# Compute salary stats per skill
skill_salary = (
    df_exploded_salary
    .groupby("skills")["salary_avg"]
    .agg(["count", "median", "mean", "min", "max"])
    .round(0)
)

# Apply the publishability threshold (n >= 5)
MIN_SAMPLE_SIZE = 5
skill_salary_publishable = (
    skill_salary[skill_salary["count"] >= MIN_SAMPLE_SIZE]
    .sort_values("median", ascending=False)
)

print(f"Skills retained (count >= {MIN_SAMPLE_SIZE}): {len(skill_salary_publishable)} / {len(skill_salary)}")
print()
skill_salary_publishable

Skills retained (count >= 5): 7 / 23



,count,median,mean,min,max
skills,,,,,
Azure,8,127400.0,134025.0,105000.0,199680.0
SQL,6,121680.0,113100.0,89440.0,128500.0
Data Pipeline,9,120000.0,112039.0,55120.0,142480.0
Machine Learning,9,95700.0,127289.0,72500.0,270750.0
Dashboards,7,92393.0,93964.0,69582.0,125000.0
Reporting,23,89877.0,91202.0,55000.0,147600.0
Predictive Modeling,5,72500.0,83640.0,65000.0,112500.0


In [29]:
# How many df_salary postings have at least one skill?
nb_with_skills = (df_salary["skills"].apply(len) > 0).sum()
nb_without_skills = (df_salary["skills"].apply(len) == 0).sum()

print(f"df_salary total: {len(df_salary)}")
print(f"  ✅ With at least 1 skill: {nb_with_skills}")
print(f"  ❌ With no skill detected: {nb_without_skills}")
print(f"  Coverage: {nb_with_skills / len(df_salary):.1%}")

df_salary total: 258
  ✅ With at least 1 skill: 61
  ❌ With no skill detected: 197
  Coverage: 23.6%


In [30]:
# Export reference tables for future sessions and README writing
salary_summary.to_csv("../outputs/salary_by_role.csv")
salary_by_city.to_csv("../outputs/salary_by_city.csv")
skill_salary_publishable.to_csv("../outputs/salary_by_skill.csv")

print("✅ 3 reference tables saved to outputs/")

✅ 3 reference tables saved to outputs/
